# Lab 06: Data skew

**Time:** about 2 h 30 min.

**Checklist questions this lab gives you evidence for:**<br><span style="color:#d93025"><b>4.1 Define data skew</b></span><br><span style="color:#d93025"><b>4.2 Identify skew in the Spark UI</b></span><br><span style="color:#d93025"><b>4.3 Why skew is dangerous</b></span><br><span style="color:#d93025"><b>4.4 Ways to solve skew</b></span><br><span style="color:#d93025"><b>4.5 Salting</b></span><br><span style="color:#d93025"><b>4.6 AQE skew join optimization</b></span><br><span style="color:#d93025"><b>4.7 Does filtering help?</b></span><br><span style="color:#d93025"><b>4.8 repartition() and skew</b></span><br><span style="color:#d93025"><b>4.9 Read-side skew</b></span><br><span style="color:#d93025"><b>4.10 repartition() vs coalesce()</b></span><br><span style="color:#d93025"><b>4.11 Iterative salting</b></span><br><span style="color:#d93025"><b>6.2 repartition() vs coalesce()</b></span>

**Goal:** find a hot key in the data and in the Spark UI, and fix a skewed join with AQE and with salting. Customer 1 (Big Reseller Ltd) places 30% of all orders. The last section is practice with windows: latest row, duplicates and streaks.

## Data information
A small online shop with **one table**. The setup cell below creates it in `data/orders.parquet` (1,000,000 orders from January to March 2024; seeded, so everyone gets the same rows):
* *order_id*; *customer_id* (empty for guest checkout), *customer_name*, *country*, *segment* (`retail`, `business` or `guest`)
* *product_id*, *category* (8 categories), *price* (price of one item)
* *quantity* (1 to 5) and *amount* = quantity × price, in dollars
* *status*: `paid`, `shipped`, `cancelled` or `returned`; *payment_method*: `card`, `cash` or `wallet`
* *order_ts* and *order_date*: when the order was placed; *tracking_code*: a 32-character code

Customer 1 is "Big Reseller Ltd": a business customer who orders in bulk (quantity 5) and places 30% of all orders. Some labs make a small lookup table from this one table (for example one row per customer); the notebook says when.

## How to work
* Write your answer in the code cell under each question and run it. The answers are at the very end of this notebook (collapsed): open one only after you tried. Each answer shows the code and the result of the reference run, so you can compare your numbers.
* When a question says *guess first*, write your guess as a comment before you run anything.
* Keep the Spark UI open (the setup cell prints its address).
* Stuck for more than 10 minutes? Read the documentation list at the top, then open the answer of that question.


## Documentation
Everything you need is on these pages. The questions do not repeat them: read, try things out in a cell, and find the answers yourself.

* [DataFrame.groupBy](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html)
* [DataFrame.repartition](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.repartition.html)
* [Spark UI: Stages tab](https://spark.apache.org/docs/latest/web-ui.html#stages-tab)
* [Optimizing skew join](https://spark.apache.org/docs/latest/sql-performance-tuning.html#optimizing-skew-join)
* [Salting (article)](https://hackernoon.com/how-to-fix-data-skew-in-apache-spark-with-the-salting-technique)
* [functions.sequence](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.sequence.html)
* [functions.countDistinct](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.count_distinct.html)
* [DataFrame.coalesce](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.coalesce.html)
* [CSV data source](https://spark.apache.org/docs/latest/sql-data-sources-csv.html)
* [Window](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Window.html)
* [functions.row_number](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.row_number.html)
* [DataFrame.dropDuplicates](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.dropDuplicates.html)
* [functions.date_sub](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.date_sub.html)

## Setup
Run the cell below once (its code is hidden: click it to expand). It creates the data and starts Spark.

In [ ]:
# Setup: run this cell once. It creates the shop data in data/ and starts Spark.
# Nothing to solve here, and you do not need to read it.
import contextlib, hashlib, io, json, os, re, shutil, sys, time, urllib.request
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this kernel's Python
from pyspark.sql import SparkSession, Window, functions as F

# ---------------------------------------------------------------- the shop data ---------
# A small online shop with ONE table, generated from seeded hash functions: every machine gets the same rows.
COUNTRIES = ["Uzbekistan", "Germany", "USA", "India", "Brazil", "Japan"]
CATEGORIES = ["books", "toys", "garden", "sports", "kitchen", "beauty", "music", "office"]
N_CUSTOMERS, N_PRODUCTS, N_ORDERS = 10_000, 200, 1_000_000

def _u(col, salt):
    """Pseudo-random number in [0, 1) computed from a column (same result on every machine)."""
    return F.pmod(F.xxhash64(col, F.lit(salt)), F.lit(1_000_000)) / 1_000_000.0

def _pick(u, options):
    return F.element_at(F.array(*[F.lit(o) for o in options]), (F.floor(u * len(options)) + 1).cast("int"))

def make_orders(spark, n=N_ORDERS, dirty=False):
    i = F.col("id")
    cust = (F.when(_u(i, 11) < 0.02, F.lit(None).cast("long"))                        # guest checkout: no customer
             .when(_u(i, 10) < 0.30, F.lit(1).cast("long"))                           # the big reseller: 30% of orders
             .otherwise((F.floor(_u(i, 12) * (N_CUSTOMERS - 1)) + 2).cast("long")))
    name = F.when(cust == 1, "Big Reseller Ltd").when(cust.isNotNull(), F.concat(F.lit("Customer "), cust.cast("string")))
    country = _pick(F.when(cust.isNotNull(), _u(cust, 1)).otherwise(_u(i, 31)), COUNTRIES)
    segment = F.when(cust.isNull(), "guest").when((cust == 1) | (_u(cust, 3) < 0.2), "business").otherwise("retail")
    product = (F.floor(_u(i, 14) * N_PRODUCTS) + 1).cast("int")
    category = F.element_at(F.array(*[F.lit(c) for c in CATEGORIES]), (product % 8 + 1).cast("int"))
    price = F.round((product * 37) % 195 + 5.99, 2)
    quantity = F.when(cust == 1, F.lit(5)).otherwise(F.floor(_u(i, 13) * 5) + 1).cast("int")   # the reseller always orders in bulk
    ts = F.lit("2024-01-01 00:00:00").cast("timestamp") + F.make_interval(secs=F.floor(_u(i, 15) * 91 * 86400).cast("int"))
    u = _u(i, 17)
    status = F.when(u < 0.55, "paid").when(u < 0.80, "shipped").when(u < 0.92, "cancelled").otherwise("returned")
    v = _u(i, 18)
    payment = F.when(v < 0.60, "card").when(v < 0.85, "cash").otherwise("wallet")
    df = spark.range(0, n, 1, 4).select(
        (i + 1).alias("order_id"), cust.alias("customer_id"), name.alias("customer_name"), country.alias("country"), segment.alias("segment"),
        product.alias("product_id"), category.alias("category"), price.alias("price"), quantity.alias("quantity"),
        F.round(quantity * price, 2).alias("amount"), status.alias("status"), payment.alias("payment_method"),
        ts.alias("order_ts"), F.md5(i.cast("string")).alias("tracking_code"))
    df = df.withColumn("order_date", F.to_date("order_ts"))
    if dirty:    # the monthly files of the final project contain mistakes on purpose
        df = df.withColumn("amount", F.when(_u(F.col("order_id"), 19) < 0.008, -F.col("amount")).otherwise(F.col("amount")))
        df = df.withColumn("quantity", F.when(_u(F.col("order_id"), 20) < 0.006, 0).otherwise(F.col("quantity")))
        df = df.unionByName(df.filter(_u(F.col("order_id"), 21) < 0.005))              # exact duplicates
    return df.select("order_id", "customer_id", "customer_name", "country", "segment", "product_id", "category", "price", "quantity",
                     "amount", "status", "payment_method", "order_ts", "order_date", "tracking_code")

def _create_data(spark):
    os.makedirs("data/monthly", exist_ok=True)
    make_orders(spark).repartition(4).write.mode("overwrite").parquet("data/orders.parquet")
    dirty = make_orders(spark, dirty=True)
    wrong = _u(F.col("order_id"), 22) < 0.003          # a few orders land in the wrong month's file
    month = F.month("order_date")
    file_month = F.when(wrong, month % 3 + 1).otherwise(month)
    for m in (1, 2, 3):
        (dirty.filter(file_month == m).coalesce(1).write.mode("overwrite")
              .parquet(f"data/monthly/orders_2024-{m:02d}.parquet"))

# ---------------------------------------------------------------- the Spark session ----
def new_session(master="local[4]", driver_memory="2g", delta=False, avro=False, aqe=False, conf=None):
    builder = (SparkSession.builder.master(master).appName("day06_lab")
               .config("spark.driver.memory", driver_memory)
               .config("spark.ui.showConsoleProgress", "false")
               .config("spark.sql.adaptive.enabled", str(aqe).lower())   # off = exact job and stage counts
               .config("spark.sql.session.timeZone", "UTC")                # same dates and hours for everyone
               .config("spark.sql.warehouse.dir", os.path.abspath("spark-warehouse"))
               # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this.
               .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false"))
    for k, v in (conf or {}).items():
        builder = builder.config(k, v)
    if delta:
        from delta import configure_spark_with_delta_pip
        builder = (builder.config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
                          .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
        builder = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.spark:spark-avro_2.13:4.2.0"] if avro else None)
    elif avro:
        builder = builder.config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.2.0")
    s = builder.getOrCreate()
    s.sparkContext.setLogLevel("ERROR")
    return s

for _d in ("spark-warehouse", "metastore_db"):
    shutil.rmtree(_d, ignore_errors=True)               # a clean catalog on every start
spark = new_session()
def _data_ok():
    """True if the single-table shop data is already there (an older three-table version is rebuilt)."""
    try:
        return "customer_name" in spark.read.parquet("data/orders.parquet").columns and os.path.exists("data/monthly/orders_2024-03.parquet")
    except Exception:
        return False

if not _data_ok():
    print("creating the shop data (about 30 seconds, once)...")
    _create_data(spark)
print(f"Spark {spark.version} | UI: {spark.sparkContext.uiWebUrl}")

# ---------------------------------------------------------------- helpers ----------------
def _tracker():
    return spark.sparkContext.statusTracker()

def _run_group(action):
    sc = spark.sparkContext
    group = f"g{time.time_ns()}"
    sc.setJobGroup(group, group)
    try:
        action()
    finally:
        sc.setLocalProperty("spark.jobGroup.id", None)
    return [_tracker().getJobInfo(j) for j in sorted(_tracker().getJobIdsForGroup(group))]

def jobs_run(action):
    """How many Spark jobs calling action() starts."""
    return len(_run_group(action))

def stages_run(action):
    """How many stages those jobs had in total."""
    return sum(len(j.stageIds) for j in _run_group(action))

def last_stage_tasks(action):
    """Number of tasks in the last stage action() ran."""
    jobs = _run_group(action)
    return _tracker().getStageInfo(max(s for j in jobs for s in j.stageIds)).numTasks

def _api(path):
    sc = spark.sparkContext
    return json.load(urllib.request.urlopen(f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}{path}"))

api = _api   # read anything from the Spark UI's REST API, e.g. api("/executors")

def _stage_rows(action):
    jobs = _run_group(action)
    ids = {s for j in jobs for s in j.stageIds}
    return [s for s in _api("/stages?length=2000") if s["stageId"] in ids]

def shuffle_bytes(action):
    """Total shuffle bytes written by the stages action() ran."""
    return sum(s["shuffleWriteBytes"] for s in _stage_rows(action))

def spill_bytes(action):
    """(memory, disk) bytes spilled by the stages action() ran."""
    rows = _stage_rows(action)
    return sum(s["memoryBytesSpilled"] for s in rows), sum(s["diskBytesSpilled"] for s in rows)

def skew_ratio(action):
    """Slowest task / median task, in the stage with the most tasks that action() ran."""
    rows = _stage_rows(action)
    st = max(rows, key=lambda s: s["numTasks"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=5000")
    times = sorted(t["taskMetrics"]["executorRunTime"] for t in tasks if t.get("taskMetrics"))
    return round(times[-1] / max(times[len(times) // 2], 1), 1)

def failed_attempts():
    """Failed task attempts in the most recent failed stage (from the Spark UI's REST API)."""
    failed = _api("/stages?status=failed")
    if not failed:
        return 0
    st = max(failed, key=lambda s: s["stageId"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=500")
    return sum(1 for t in tasks if t["status"] == "FAILED")

def sql_metric(action, metric):
    """Value of one scan metric (for example 'number of files read') of the query action() runs."""
    desc = f"q{time.time_ns()}"
    spark.sparkContext.setJobDescription(desc)
    try:
        action()
    finally:
        spark.sparkContext.setJobDescription(None)
    for ex in _api("/sql?details=true&planDescription=false&length=2000"):
        if ex.get("description") == desc:
            vals = [m["value"] for n in ex["nodes"] for m in n["metrics"] if m["name"] == metric]
            return vals[0] if vals else None

def noop(df):
    """Run the whole query but throw the result away."""
    df.write.format("noop").mode("overwrite").save()

def plan(df, mode="formatted"):
    """explain() output as a string."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode)
    return buf.getvalue()

def count_nodes(df, *names):
    """How often each operator name appears in the simple plan."""
    text = plan(df, "simple")
    return {n: len(re.findall(r"\b" + re.escape(n) + r"\b", text)) for n in names}

def optimized(df):
    """The optimized logical plan as a string."""
    return df._jdf.queryExecution().optimizedPlan().toString()

def operators(df):
    """Operator names of the physical plan tree, top to bottom."""
    tree = plan(df).split("\n\n")[0].splitlines()[1:]
    return [re.sub(r"\s*\(\d+\)$", "", line).strip(" +-*:") for line in tree]

def rules_fired(df):
    """Names of the Catalyst rules that changed this query's plan."""
    qe = df._jdf.queryExecution(); qe.executedPlan()
    it, fired = qe.tracker().rules().iterator(), set()
    while it.hasNext():
        kv = it.next()
        if kv._2().numEffectiveInvocations() > 0:
            fired.add(kv._1().split(".")[-1].split("$")[-1])
    return fired

def estimate(df):
    """(sizeInBytes, rowCount or None) the optimizer estimates for df."""
    stats = df._jdf.queryExecution().optimizedPlan().stats()
    return int(str(stats.sizeInBytes())), (int(str(stats.rowCount().get())) if stats.rowCount().isDefined() else None)

def partition_sizes(df):
    """Rows per partition, largest first."""
    return sorted((r["count"] for r in df.groupBy(F.spark_partition_id().alias("p")).count().collect()), reverse=True)

def timeit(action):
    start = time.perf_counter(); action(); return round(time.perf_counter() - start, 2)

def folder_mb(path):
    files = [os.path.join(d, f) for d, _, fs in os.walk(path) for f in fs] if os.path.isdir(path) else [path]
    return round(sum(os.path.getsize(f) for f in files) / 1e6, 1)


Load the data the way Lab 01 taught you (given):

In [ ]:
orders = spark.read.parquet("data/orders.parquet")

# a small lookup table made from the same table: one row per customer (given)
(orders.filter(F.col("customer_id").isNotNull())
       .select("customer_id", F.col("country").alias("home_country"), F.col("segment").alias("customer_segment")).distinct()
       .write.mode("overwrite").parquet("data/customers_lookup.parquet"))
customers = spark.read.parquet("data/customers_lookup.parquet")
orders.createOrReplaceTempView("orders")          # so SQL can use the tables too
customers.createOrReplaceTempView("customers")

# for this lab: no broadcast joins (they would hide the skew), and 64 shuffle partitions
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.shuffle.partitions", 64)

## A. Find the skew

**6.1 Which customer has the most orders, and how big is the skew?** Count the orders per *customer_id* (guests have `null`).
* *top3* = the 3 largest groups as `[(customer_id, orders), ...]`, largest first
* *share* = the share of the largest group in all orders, in percent (1 decimal)

Put `[top3, share]` in *result*.

**6.2 How uneven are the partitions after `orders.repartition(64, "customer_id")`?** Use `partition_sizes` (rows per partition, largest first). *ratio* = the largest size divided by the median size, rounded to 1 decimal.

## B. A skewed join, and four fixes

**6.3 Find the straggler.** Run `plain = orders.join(customers, "customer_id")` with `noop` and measure *skew_plain* = `skew_ratio(...)`: the slowest task divided by the median task in the join stage. Look at the same stage in the Spark UI (Stages tab: summary metrics, event timeline).

**6.4 Let AQE split the hot partition.** Turn AQE on, set `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` and `spark.sql.adaptive.advisoryPartitionSizeInBytes` to `4MB` (our data is small), and run two queries on the join: grouped by *home_country* with `count("tracking_code")` (*with_payload*) and with `count("customer_id")` (*key_only*). Build each, `collect()` it and look for `skew=true` in the final plan (`plan(q, "simple")`, before `== Initial Plan ==`). Put `[with_payload_detected, key_only_detected]` in *result* (True/False each), then turn AQE off. Why is one not detected?

**6.5 Fix it with salting.** Use 16 salt values.
* add *salt* to *orders*: a number from 0 to 15 computed from the order (`F.pmod(F.xxhash64("order_id"), F.lit(16))`)
* repeat every customer 16 times, once per salt (`F.explode(F.sequence(...))`)
* join on `["customer_id", "salt"]` and drop the salt

Measure *skew_salted* and compare the row counts of both joins (*same_rows*, True/False).

**6.6 Salt only the hot key (iterative salting).** Give salt values 0 to 15 only to customer 1; every other customer gets salt 0. Build *orders_h* and *customers_h* (customer 1 repeated 16 times, every other customer once), put the number of rows of *customers_h* in *rows_h*, and measure *skew_hot* of the join.

**6.7 A skewed aggregation: distinct products per customer.** *exact* = dict `{customer_id: number of distinct product_id}` from one `groupBy` with `countDistinct`. Then do it in two stages: add a salt `F.pmod(F.xxhash64("product_id"), F.lit(16))`, count distinct products per (customer, salt), then add the counts per customer (*two_stage*, same dict). They must be equal (*equal*). What is the value for customer 1 (*hot_value*)?

## C. repartition, coalesce and the read side

**6.8 What do repartition and coalesce do to partition sizes?** Put the 3 largest partition sizes (`partition_sizes(df)[:3]`) of each DataFrame in a dict *sizes*:
* `even`: `orders.repartition(8)`
* `by_key`: `orders.repartition(8, "customer_id")`
* `coalesced`: `orders.repartition(8).coalesce(2)`
* `by_key_coalesced`: `orders.repartition(8, "customer_id").coalesce(2)`

**6.9 Skew on the read side.** Write *orders* as one gzip-compressed CSV (`coalesce(1)`, header, folder `skew_gz`) and read it back as *gz*. How many read partitions does it have (*gzip_parts*)? Fix the next stage with `repartition(8)` and put the 3 largest partition sizes in *after*. Put `[gzip_parts, after]` in *result*.

## D. Coding practice: latest row, duplicates and streaks

Three problems with `Window`. Break ties explicitly so your answer is always the same.

**6.10 What was the most recent order of customers 2, 3 and 4?** Use the latest *order_ts* (ties by the larger *order_id*). *latest* = list of `(customer_id, order_id, amount, status)` sorted by *customer_id*.

**6.11 Remove duplicate clicks.** *clicks* (given) has duplicated events. Keep only the earliest row (by *ts*) for every pair (*user*, *action*). *deduped* = list of `(user, ts, action)` sorted by *user*, then *action*.

In [ ]:
clicks = spark.createDataFrame([
    ("u1", "2024-03-01 10:00:00", "view"), ("u1", "2024-03-01 10:00:05", "view"), ("u1", "2024-03-01 10:01:00", "buy"),
    ("u2", "2024-03-01 11:00:00", "view"), ("u2", "2024-03-01 11:00:00", "view"), ("u2", "2024-03-01 11:30:00", "buy"),
    ("u3", "2024-03-02 09:00:00", "view"),
], "user string, ts string, action string")


**6.12 What is the longest streak of consecutive order days of customers 2 to 6?** A day counts when the customer placed at least one order. Hint: take the distinct days per customer, subtract the row number (in date order) from the date; days of one streak get the same value. *streaks* = list of `(customer_id, longest streak in days)` sorted by *customer_id*.

---
Done? In the Theory part of Day 6, answer the red checklist topics aloud, with the ratios and plans from this lab as evidence.

## Answers
Try every question yourself first. When you are stuck, click the arrow of that question below to open its answer, then compare it with your own. Each answer is the reference solution with the result of its run; copy the code into the cell above to run it.

<details>
<summary><b>6.1</b> &nbsp;Which customer has the most orders, and how big is the skew?</summary>

```python
counts = orders.groupBy("customer_id").count().orderBy(F.desc("count"))
top3 = [(r["customer_id"], r["count"]) for r in counts.limit(3).collect()]
share = round(100 * top3[0][1] / orders.count(), 1)
result = [top3, share]
result
```

Result:

```
[[(1, 293049), (None, 20011), (6518, 101)], 29.3]
```

</details>

<details>
<summary><b>6.2</b> &nbsp;How uneven are the partitions after orders.repartition(64, "customer_id")?</summary>

```python
sizes = partition_sizes(orders.repartition(64, "customer_id"))
ratio = round(sizes[0] / sizes[len(sizes) // 2], 1)
ratio
```

Result:

```
28.2
```

</details>

<details>
<summary><b>6.3</b> &nbsp;Find the straggler.</summary>

```python
plain = orders.join(customers, "customer_id")
skew_plain = skew_ratio(lambda: noop(plain))
skew_plain
```

Result:

```
12.7
```

</details>

<details>
<summary><b>6.4</b> &nbsp;Let AQE split the hot partition.</summary>

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "4MB")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "4MB")

def detected(agg):
    q = orders.join(customers, "customer_id").groupBy("home_country").agg(agg)
    q.collect()
    return "skew=true" in plan(q, "simple").split("== Initial Plan ==")[0]

result = [detected(F.count("tracking_code")), detected(F.count("customer_id"))]
spark.conf.set("spark.sql.adaptive.enabled", "false")
result
```

Result:

```
[True, False]
```

</details>

<details>
<summary><b>6.5</b> &nbsp;Fix it with salting.</summary>

```python
N = 16
orders_salted = orders.withColumn("salt", F.pmod(F.xxhash64("order_id"), F.lit(N)).cast("int"))
customers_x = customers.withColumn("salt", F.explode(F.sequence(F.lit(0), F.lit(N - 1))))
salted = orders_salted.join(customers_x, ["customer_id", "salt"]).drop("salt")
skew_salted = skew_ratio(lambda: noop(salted))
same_rows = salted.count() == plain.count()
skew_plain, skew_salted, same_rows
```

Result:

```
(12.7, 3.4, True)
```

</details>

<details>
<summary><b>6.6</b> &nbsp;Salt only the hot key (iterative salting).</summary>

```python
hot = F.col("customer_id") == 1
orders_h = orders.withColumn("salt", F.when(hot, F.pmod(F.xxhash64("order_id"), F.lit(N))).otherwise(0).cast("int"))
customers_h = customers.withColumn("salt", F.explode(F.when(hot, F.sequence(F.lit(0), F.lit(N - 1))).otherwise(F.array(F.lit(0)))))
rows_h = customers_h.count()
skew_hot = skew_ratio(lambda: noop(orders_h.join(customers_h, ["customer_id", "salt"])))
rows_h, skew_hot
```

Result:

```
(10015, 3.9)
```

</details>

<details>
<summary><b>6.7</b> &nbsp;A skewed aggregation: distinct products per customer.</summary>

```python
exact = {r["customer_id"]: r["n"] for r in orders.groupBy("customer_id").agg(F.countDistinct("product_id").alias("n")).collect()}
stage1 = (orders.withColumn("salt", F.pmod(F.xxhash64("product_id"), F.lit(16)))
          .groupBy("customer_id", "salt").agg(F.countDistinct("product_id").alias("c")))
two_stage = {r["customer_id"]: r["n"] for r in stage1.groupBy("customer_id").agg(F.sum("c").alias("n")).collect()}
equal = exact == two_stage
hot_value = exact[1]
equal, hot_value
```

Result:

```
(True, 200)
```

</details>

<details>
<summary><b>6.8</b> &nbsp;What do repartition and coalesce do to partition sizes?</summary>

```python
sizes = {
    "even": partition_sizes(orders.repartition(8))[:3],
    "by_key": partition_sizes(orders.repartition(8, "customer_id"))[:3],
    "coalesced": partition_sizes(orders.repartition(8).coalesce(2))[:3],
    "by_key_coalesced": partition_sizes(orders.repartition(8, "customer_id").coalesce(2))[:3],
}
sizes
```

Result:

```
{'even': [125000, 125000, 125000],
 'by_key': [378795, 104509, 88721],
 'coalesced': [500000, 500000],
 'by_key_coalesced': [636566, 363434]}
```

</details>

<details>
<summary><b>6.9</b> &nbsp;Skew on the read side.</summary>

```python
shutil.rmtree("skew_gz", ignore_errors=True)
orders.coalesce(1).write.option("header", True).option("compression", "gzip").csv("skew_gz")
gz = spark.read.option("header", True).csv("skew_gz")
gzip_parts = gz.rdd.getNumPartitions()
after = partition_sizes(gz.repartition(8))[:3]
result = [gzip_parts, after]
result
```

Result:

```
[1, [125000, 125000, 125000]]
```

</details>

<details>
<summary><b>6.10</b> &nbsp;What was the most recent order of customers 2, 3 and 4?</summary>

```python
w = Window.partitionBy("customer_id").orderBy(F.desc("order_ts"), F.desc("order_id"))
last = (orders.filter(F.col("customer_id").between(2, 4))
        .withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1))
latest = [(r["customer_id"], r["order_id"], r["amount"], r["status"]) for r in last.orderBy("customer_id").collect()]
latest
```

Result:

```
[(2, 78284, 323.98, 'shipped'),
 (3, 242330, 603.96, 'paid'),
 (4, 294213, 575.97, 'cancelled')]
```

</details>

<details>
<summary><b>6.11</b> &nbsp;Remove duplicate clicks.</summary>

```python
clicks = spark.createDataFrame([
    ("u1", "2024-03-01 10:00:00", "view"), ("u1", "2024-03-01 10:00:05", "view"), ("u1", "2024-03-01 10:01:00", "buy"),
    ("u2", "2024-03-01 11:00:00", "view"), ("u2", "2024-03-01 11:00:00", "view"), ("u2", "2024-03-01 11:30:00", "buy"),
    ("u3", "2024-03-02 09:00:00", "view"),
], "user string, ts string, action string")

w = Window.partitionBy("user", "action").orderBy("ts")
kept = clicks.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1)
deduped = [(r["user"], r["ts"], r["action"]) for r in kept.orderBy("user", "action").collect()]
deduped
```

Result:

```
[('u1', '2024-03-01 10:01:00', 'buy'),
 ('u1', '2024-03-01 10:00:00', 'view'),
 ('u2', '2024-03-01 11:30:00', 'buy'),
 ('u2', '2024-03-01 11:00:00', 'view'),
 ('u3', '2024-03-02 09:00:00', 'view')]
```

</details>

<details>
<summary><b>6.12</b> &nbsp;What is the longest streak of consecutive order days of customers 2 to 6?</summary>

```python
days = orders.filter(F.col("customer_id").between(2, 6)).select("customer_id", "order_date").distinct()
w = Window.partitionBy("customer_id").orderBy("order_date")
grouped = days.withColumn("grp", F.date_sub("order_date", F.row_number().over(w).cast("int")))
longest = (grouped.groupBy("customer_id", "grp").count()
           .groupBy("customer_id").agg(F.max("count").alias("longest")).orderBy("customer_id"))
streaks = [(r["customer_id"], r["longest"]) for r in longest.collect()]
streaks
```

Result:

```
[(2, 7), (3, 6), (4, 4), (5, 10), (6, 5)]
```

</details>